# Datenanalyse mit SQL & Python - Tag 4: Übungen

**Teil A:** Warm-up mit Shop-Daten als Recap zu Tag 3  
**Teil B:** Cleaning, EDA & Reporting mit messy HR Data  
**Mini-Projekt:** Welche Faktoren hängen mit Gehaltsstruktur oder Fluktuationsrisiko im HR-Datensatz zusammen?


## Lernziele

In diesen Übungen trainierst du:

- Inhalte vom Vortag mit Shop-Daten zu wiederholen: Pandas, `merge`, `groupby`, Visualisierung
- einen messy HR-Datensatz zu prüfen und zu bereinigen
- HR-Kennzahlen mit Pandas und SQL zu analysieren
- Verteilungen, Gruppenunterschiede und Korrelationen zu visualisieren
- aus HR-Daten eine datenbasierte Handlungsempfehlung abzuleiten


## Einrichtung & Daten laden

Führe diese Zelle zuerst aus. Sie lädt:

- `shop_joined` für den Warm-up
- `hr_raw` aus dem Messy-dataset Repository
- `hr` als Arbeitskopie für Cleaning und EDA


In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 100)


def find_shop_dir():
    candidates = [
        Path('data_practice_tables/shop'),
        Path('Tag 1/data_practice_tables/shop'),
        Path('../Tag 1/data_practice_tables/shop'),
    ]
    for candidate in candidates:
        if (candidate / 'shop_orders.csv').exists():
            return candidate
    raise FileNotFoundError('Shop-Daten wurden nicht gefunden.')

SHOP_DIR = find_shop_dir()
orders = pd.read_csv(SHOP_DIR / 'shop_orders.csv')
customers = pd.read_csv(SHOP_DIR / 'shop_customers.csv')
products = pd.read_csv(SHOP_DIR / 'shop_products.csv')

shop_joined = orders.merge(customers, on='customer_id', how='left').merge(products, on='product_id', how='left')
shop_joined['revenue'] = shop_joined['quantity'] * shop_joined['price']

HR_DATA_URL = 'https://raw.githubusercontent.com/eyowhite/Messy-dataset/main/messy_HR_data.csv'
hr_raw = pd.read_csv(HR_DATA_URL, engine='python', on_bad_lines='warn')
hr = hr_raw.copy()

print('shop_joined:', shop_joined.shape)
print('hr_raw:', hr_raw.shape)
hr_raw.head()


## Teil A: Warm-up mit Shop-Daten als Recap zu Tag 3

Arbeite mit `shop_joined`. Ziel ist, die wichtigsten Inhalte vom Vortag in 25-30 Minuten zu wiederholen.


### Übung A1: DataFrame-Überblick (ca. 4 Minuten)

**Aufgabe:** Zeige die ersten Zeilen, Spaltennamen und Datentypen von `shop_joined`.


In [ ]:
shop_joined._____()


In [ ]:
shop_joined._____


In [ ]:
shop_joined._____()


### Übung A2: Umsatz pro Stadt (ca. 6 Minuten)

**Aufgabe:** Berechne den Gesamtumsatz pro Stadt.


In [ ]:
revenue_by_city = (
    shop_joined
    .groupby('_____')['_____']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_city


### Übung A3: Warm-up-Visualisierung (ca. 8 Minuten)

**Aufgabe:** Visualisiere den Umsatz pro Produktkategorie.


In [ ]:
revenue_by_category = (
    shop_joined
    .groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
)
plot_data = revenue_by_category.reset_index()
plot_data.columns = ['category', 'revenue']

sns.barplot(data=plot_data, x='revenue', y='category')
plt.title('Warm-up: Umsatz pro Produktkategorie')
plt.xlabel('Umsatz')
plt.ylabel('Kategorie')
plt.show()


## Teil B: Cleaning, EDA & Reporting mit messy HR Data

Arbeite mit `hr_raw` und `hr`. Der Datensatz stammt aus dem Repository `eyowhite/Messy-dataset`.

**Business-Kontext:** Die HR-Abteilung möchte verstehen, welche Faktoren mit Gehaltsstruktur, Zufriedenheit oder möglichem Fluktuationsrisiko zusammenhängen.


### Übung B1: Schnell-Check

**Aufgabe:** Prüfe Form, Spalten, Datentypen, fehlende Werte und Duplikate.


In [ ]:
print('Form:', hr_raw.shape)
print()
print('Spalten:')
print(hr_raw.columns.tolist())
print()
print('Datentypen:')
print(hr_raw.dtypes)
print()
print('Fehlende Werte:')
print(hr_raw.isna().sum().sort_values(ascending=False).head(20))
print()
print('Duplikate:', hr_raw.duplicated().sum())


### Übung B2: Spalten und Textwerte bereinigen

**Aufgabe:** Vereinheitliche Spaltennamen, entferne Leerzeichen aus Textspalten und wandle leere Strings in `NA` um.


In [ ]:
hr = hr_raw.copy()

hr.columns = (
    hr.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

text_cols = hr.select_dtypes(include=['object']).columns
for col in text_cols:
    hr[col] = hr[col].astype('string').str.strip()

hr = hr.replace(r'^\s*$', pd.NA, regex=True)
hr.head()


### Übung B3: Zahlen konvertieren

**Aufgabe:** Suche numerische Spalten, die als Text gespeichert sind, und konvertiere geeignete Spalten mit `pd.to_numeric`.

**Hinweis:** Passe die Spaltennamen an, nachdem du `hr.columns` geprüft hast.


In [ ]:
# Beispiel: Konvertiere alle Spalten, die wahrscheinlich numerisch sind.
for col in hr.columns:
    maybe_numeric = hr[col].astype('string').str.replace('.', '', regex=False).str.replace(',', '', regex=False)
    numeric_share = pd.to_numeric(maybe_numeric, errors='coerce').notna().mean()
    if numeric_share > 0.7:
        hr[col] = pd.to_numeric(maybe_numeric, errors='coerce')

hr.dtypes


### Übung B4: Fehlende Werte und Duplikate nach Cleaning prüfen

**Aufgabe:** Vergleiche den Zustand vor und nach den ersten Cleaning-Schritten.


In [ ]:
missing_report = pd.DataFrame({
    'missing_before': hr_raw.isna().sum(),
    'missing_after': hr.isna().sum()
}).fillna(0).astype(int)

print('Form vorher:', hr_raw.shape)
print('Form nachher:', hr.shape)
print('Duplikate nach Cleaning:', hr.duplicated().sum())
missing_report.sort_values('missing_after', ascending=False).head(15)


### B5: Explorative HR-Analyse

Wähle passende Spalten aus dem Datensatz. Je nach tatsächlicher Spaltenstruktur kannst du die folgenden Beispiele anpassen.


### Übung B5: Numerische Kennzahlen

**Aufgabe:** Berechne Kennzahlen für alle numerischen Spalten.


In [ ]:
numeric_hr = hr.select_dtypes(include='number')
numeric_hr.describe().T


### Übung B6: Verteilung einer wichtigen HR-Kennzahl

**Aufgabe:** Wähle eine numerische Spalte, zum Beispiel Gehalt, Alter, Zufriedenheit oder Performance, und visualisiere die Verteilung.


In [ ]:
# Wähle eine numerische Spalte aus der Ausgabe von numeric_hr.columns
numeric_hr.columns


In [ ]:
selected_numeric_col = numeric_hr.columns[0]

plt.hist(hr[selected_numeric_col].dropna(), bins=20)
plt.title(f'Verteilung: {selected_numeric_col}')
plt.xlabel(selected_numeric_col)
plt.ylabel('Anzahl')
plt.show()


### Übung B7: Gruppenvergleich

**Aufgabe:** Wähle eine kategoriale Spalte und vergleiche den Durchschnitt einer numerischen HR-Kennzahl pro Gruppe.


In [ ]:
categorical_cols = hr.select_dtypes(include=['object', 'string']).columns
categorical_cols


In [ ]:
selected_group_col = categorical_cols[0]
selected_value_col = numeric_hr.columns[0]

group_report = (
    hr
    .groupby(selected_group_col)[selected_value_col]
    .agg(['count', 'mean', 'median'])
    .sort_values('mean', ascending=False)
)
group_report.head(10)


### Übung B8: Korrelationen

**Aufgabe:** Erstelle eine Korrelationsmatrix für numerische HR-Spalten.


In [ ]:
corr = numeric_hr.corr()

sns.heatmap(corr, annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Korrelationen im HR-Datensatz')
plt.show()


### B6: SQL + Python für HR-Reporting

Speichere den bereinigten HR-DataFrame in SQLite und erstelle eine aggregierte Tabelle.


In [ ]:
conn = sqlite3.connect(':memory:')
hr.to_sql('hr', conn, index=False, if_exists='replace')

# Passe group/value bei Bedarf an deine Spalten an.
group_col = selected_group_col
value_col = selected_value_col

query = f'''
SELECT
    "{group_col}" AS gruppe,
    COUNT(*) AS anzahl,
    AVG("{value_col}") AS durchschnitt
FROM hr
WHERE "{value_col}" IS NOT NULL
GROUP BY "{group_col}"
HAVING COUNT(*) >= 2
ORDER BY durchschnitt DESC;
'''

sql_hr_report = pd.read_sql_query(query, conn)
sql_hr_report.head(10)


### B7: Mini-Projekt HR Business Question

**Business Question:**

> Welche Faktoren hängen im HR-Datensatz mit Gehaltsstruktur, Zufriedenheit oder möglichem Fluktuationsrisiko zusammen?

Wähle eine konkrete Variante:

- Welche Abteilung oder Rolle hat auffällig hohe/niedrige Gehälter?
- Gibt es Gruppen mit niedriger Zufriedenheit oder Performance?
- Welche Merkmale könnten auf Fluktuationsrisiko hinweisen?
- Welche HR-Maßnahme würdest du empfehlen?

**Ergebnis:** 2 Diagramme, 3 Erkenntnisse, 1 Empfehlung und 1 Hinweis zur Datenqualität.


In [ ]:
hr_insights = [
    'Erkenntnis 1: ...',
    'Erkenntnis 2: ...',
    'Erkenntnis 3: ...',
]
recommendation = 'Empfehlung: ...'
quality_note = 'Datenqualitäts-Hinweis: ...'

for insight in hr_insights:
    print('-', insight)
print(recommendation)
print(quality_note)


## Abschlussreflexion

Diskutiert kurz:

1. Welche Cleaning-Entscheidung war für die Analyse am wichtigsten?
1. Welche HR-Frage ließ sich gut beantworten?
1. Welche Aussage wäre wegen Datenqualität noch unsicher?
1. Welche zusätzliche HR-Spalte wäre für eine bessere Analyse hilfreich?
